ModuleNotFoundError: No module named 'imblearn'

# Predicting Bank Term Deposit Subscription Using Machine Learning

## Project Overview

### Business Problem
Banks spend time and money contacting customers during marketing campaigns. The objective of this project is to predict whether a customer is likely to subscribe to a term deposit, so the marketing team can prioritize higher-probability prospects.

### Machine Learning Problem
This is a **Binary Classification** problem because the target variable `y` has two possible values:

- `yes`: the customer subscribed to a term deposit
- `no`: the customer did not subscribe

### Production Design Decision
The production-oriented model excludes `duration` because call duration is only known after the call ends. Using it for pre-call targeting would create data leakage.

### Project Workflow
`Business Understanding → Data Quality → EDA → Preprocessing → Model Comparison → Evaluation → Explainability → Deployment`

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

In [ ]:
DATA_PATH = Path("data/bank-additional-full.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset was not found at: {DATA_PATH.resolve()}\n"
        "Make sure bank-additional-full.csv is inside the data folder."
    )

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Dataset loaded successfully ✅")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.head()

## 2. Data Understanding

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset shape:")
print(df.shape)

display(df.head())

### 2.1 Feature Groups

| Group | Variables | Purpose |
|---|---|---|
| Customer profile | `age`, `job`, `marital`, `education`, `default`, `housing`, `loan` | Describe the customer |
| Campaign context | `contact`, `month`, `day_of_week`, `duration`, `campaign`, `pdays`, `previous`, `poutcome` | Describe contact history and campaign activity |
| Economic indicators | `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed` | Capture broader economic conditions |
| Target | `y` | Indicates whether the customer subscribed (`yes` / `no`) |

## 3. Data Quality Assessment

In [ ]:
quality_report = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows",
        "Target Classes"
    ],
    "Result": [
        df.shape[0],
        df.shape[1],
        df.isnull().sum().sum(),
        df.duplicated().sum(),
        df["y"].nunique()
    ]
})

quality_report

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)

print("Missing values by column:")
display(missing_values)

print(f"Duplicate rows: {df.duplicated().sum():,}")

print("\nTarget distribution:")
display(df["y"].value_counts())

print("\nTarget percentage:")
display((df["y"].value_counts(normalize=True) * 100).round(2))

### 3.1 Data Quality Decision Log

- **Missing values:** no missing values were found in the source data.
- **Duplicate rows:** 12 duplicate rows were identified. They are retained because the dataset has no unique customer identifier, so identical rows may still represent separate real campaign records.
- **Target balance:** the positive class is relatively rare, so the evaluation will include Precision, Recall, F1-score, and ROC-AUC rather than Accuracy alone.

## 4. Data Preparation

In [ ]:
df_clean = df.copy()

# تنظيف المسافات قبل وبعد النصوص
categorical_columns_all = df_clean.select_dtypes(include="object").columns

for col in categorical_columns_all:
    df_clean[col] = df_clean[col].str.strip()

#التأكد أن الـTarget يحتوي فقط على yes/no
print(df_clean["y"].value_counts())

# تحويل Target إلى 0 و1
df_clean["y_num"] = df_clean["y"].map({
    "no": 0,
    "yes": 1
})

print("\nTarget after encoding:")
display(df_clean["y_num"].value_counts())

### 4.1 Leakage Control and Production Feature Set

The model target is `y_num`. The feature `duration` is excluded because it is only available after the call is completed. This preserves a realistic pre-call targeting scenario.

In [ ]:
TARGET = "y_num"

LEAKAGE_FEATURES = [
    "duration"
]

FEATURES = [
    col for col in df_clean.columns
    if col not in ["y", "y_num"] + LEAKAGE_FEATURES
]

print(f"Number of usable features: {len(FEATURES)}")
print(FEATURES)

## 5. Exploratory Data Analysis (EDA)

**EDA objective:** explore how customer profile, campaign characteristics, and economic indicators are associated with term-deposit subscription.

> All findings in this section describe observed associations in the historical data. They do not establish causation.

### 5.1 Target Distribution

In [ ]:
target_summary = (
    df_clean["y"]
    .value_counts()
    .reset_index()
)

target_summary.columns = ["Subscription", "Customers"]

fig = px.bar(
    target_summary,
    x="Subscription",
    y="Customers",
    text="Customers",
    title="Distribution of Term Deposit Subscription"
)

fig.update_traces(textposition="outside")
fig.show()

**Insight — Class imbalance:** `no` represents 36,548 customers (88.73%) and `yes` represents 4,640 customers (11.27%). Because the positive class is much smaller, accuracy alone is not sufficient for model evaluation.

### 5.2 Age Distribution by Subscription Outcome

In [ ]:
fig = px.histogram(
    df_clean,
    x="age",
    color="y",
    barmode="overlay",
    nbins=40,
    histnorm="percent",
    title="Age Distribution by Subscription Outcome"
)

fig.show()

**Insight:** The 30–34 age band contains the largest share of subscription outcomes in this histogram (21.5%). This describes the distribution of subscribers, not the subscription rate of every age group.

### 5.3 Subscription Rate by Job

In [ ]:
job_subscription = (
    df_clean
    .groupby("job", as_index=False)["y_num"]
    .mean()
    .sort_values("y_num", ascending=False)
)

job_subscription["subscription_rate"] = job_subscription["y_num"] * 100

fig = px.bar(
    job_subscription,
    x="job",
    y="subscription_rate",
    text="subscription_rate",
    title="Subscription Rate by Job"
)

fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(xaxis_tickangle=-45)
fig.show()

**Insight:** Students have the highest observed subscription rate (31.43%), while blue-collar customers have the lowest rate (6.89%). This is an observed association and may also be influenced by the number of customers in each job category.

### 5.4 Subscription Rate by Contact Type

In [ ]:
contact_subscription = (
    df_clean
    .groupby("contact", as_index=False)["y_num"]
    .mean()
)

contact_subscription["subscription_rate"] = contact_subscription["y_num"] * 100

fig = px.bar(
    contact_subscription,
    x="contact",
    y="subscription_rate",
    text="subscription_rate",
    title="Subscription Rate by Contact Type"
)

fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()

**Insight:** Cellular contact has a higher observed subscription rate (14.74%) than telephone contact (5.23%). This makes contact channel a potentially useful predictive feature.

### 5.5 Subscription Rate by Campaign Month

In [ ]:
month_order = [
    "mar", "apr", "may", "jun", "jul", "aug",
    "sep", "oct", "nov", "dec"
]

month_subscription = (
    df_clean
    .groupby("month", as_index=False)["y_num"]
    .mean()
)

month_subscription["subscription_rate"] = month_subscription["y_num"] * 100
month_subscription["month"] = pd.Categorical(
    month_subscription["month"],
    categories=month_order,
    ordered=True
)

month_subscription = month_subscription.sort_values("month")

fig = px.bar(
    month_subscription,
    x="month",
    y="subscription_rate",
    text="subscription_rate",
    title="Subscription Rate by Month"
)

fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()

**Insight:** March (50.55%), December (48.90%), September (44.91%), and October (43.87%) have the highest observed subscription rates. These values should be interpreted alongside each month’s sample size.

### 5.6 Subscription Rate by Previous Campaign Outcome

In [ ]:
poutcome_subscription = (
    df_clean
    .groupby("poutcome", as_index=False)["y_num"]
    .mean()
    .sort_values("y_num", ascending=False)
)

poutcome_subscription["subscription_rate"] = (
    poutcome_subscription["y_num"] * 100
)

fig = px.bar(
    poutcome_subscription,
    x="poutcome",
    y="subscription_rate",
    text="subscription_rate",
    title="Subscription Rate by Previous Campaign Outcome"
)

fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.show()

**Insight:** Customers with a previous campaign outcome of `success` have a 65.11% observed subscription rate, compared with 14.23% after `failure` and 8.83% when no previous outcome exists. This makes `poutcome` a strong predictive signal.

### 5.7 Subscription Rate by Number of Contacts

In [ ]:
campaign_summary = (
    df_clean
    .groupby("campaign", as_index=False)["y_num"]
    .mean()
)

campaign_summary["subscription_rate"] = campaign_summary["y_num"] * 100

campaign_summary = campaign_summary[campaign_summary["campaign"] <= 10]

fig = px.line(
    campaign_summary,
    x="campaign",
    y="subscription_rate",
    markers=True,
    title="Subscription Rate by Number of Contacts in Current Campaign"
)

fig.show()

**Insight:** Subscription rate generally declines as the number of contacts increases from one to eight. The variation at higher contact counts should be interpreted carefully because those groups may contain fewer customers.

### 5.8 Key EDA Insights

1. The target is imbalanced: 88.73% of customers did not subscribe and 11.27% subscribed.
2. Subscription rates differ materially across job categories; students show the highest observed rate.
3. Cellular contact is associated with a higher observed subscription rate than telephone contact.
4. A successful previous campaign outcome is strongly associated with future subscription.
5. Campaign month and the number of contacts are associated with different observed conversion rates.
6. These relationships are predictive associations in historical data, not causal claims.

## 6. Define the Feature Matrix (X) and Target Vector (y)

In [ ]:
X = df_clean[FEATURES].copy()
y = df_clean[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

## 7. Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

## 8. Identify Numeric and Categorical Features

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

## 9. Preprocessing Pipeline

Numeric and categorical variables require different preparation. The `ColumnTransformer` applies the correct transformations to each group, and the `Pipeline` ensures that preprocessing is learned from training data only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
numeric_transformer = ImbPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = ImbPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 10. Candidate Models

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
class_weight_setting = "balanced" if y.mean() < 0.40 else None

models = {
    "Dummy Baseline": DummyClassifier(strategy="prior"),
    
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight=class_weight_setting,
        random_state=42
    ),
    
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10,
        min_samples_leaf=20,
        class_weight=class_weight_setting,
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=5,
        class_weight=class_weight_setting,
        random_state=42,
        n_jobs=-1
    )
}

## 11. Cross-Validation and Model Comparison

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [ ]:
pipelines = {}
comparison_rows = []

for model_name, model in models.items():
    
    pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
    ("model", model)
    ])
    
    pipelines[model_name] = pipeline
    
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    
    comparison_rows.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1 Score": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

comparison = pd.DataFrame(comparison_rows)

comparison = comparison.sort_values(
    by=["CV ROC-AUC", "CV F1 Score"],
    ascending=False
).reset_index(drop=True)

comparison.round(4)

In [ ]:
# Visual comparison of cross-validation metrics
comparison_long = comparison.melt(
    id_vars="Model",
    value_vars=[
        "CV Accuracy",
        "CV Precision",
        "CV Recall",
        "CV F1 Score",
        "CV ROC-AUC"
    ],
    var_name="Metric",
    value_name="Score"
)

fig = px.bar(
    comparison_long,
    x="Model",
    y="Score",
    color="Metric",
    barmode="group",
    title="Cross-Validation Model Comparison"
)

fig.update_yaxes(range=[0, 1], title="Mean Cross-Validation Score")
fig.update_xaxes(title="")
fig.show()

### 11.1 Model Comparison Interpretation

Although the Dummy Baseline has the highest Accuracy (0.8873), it predicts only the majority class and has **zero Precision, Recall, and F1-score for subscribers**. It is therefore not useful for the business objective.

Random Forest is selected because it has the strongest cross-validation ROC-AUC (0.7952) and F1-score (0.4799) among the tested models, while still detecting a meaningful share of likely subscribers.

## 12. Select, Train, and Evaluate the Best Model

The best model is selected dynamically using **mean cross-validation ROC-AUC** as the primary criterion and **F1-score** as a secondary criterion. The selected model is then fitted on the complete training set and evaluated once on the untouched test set.

In [ ]:
best_model_name = comparison.iloc[0]["Model"]

print(f"Best model based on CV ROC-AUC: {best_model_name}")

best_pipeline = pipelines[best_model_name]

best_pipeline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve
)

THRESHOLD = 0.50

# Retrieve the probability of the positive class (1 = subscription) safely.
positive_class_index = list(best_pipeline.classes_).index(1)
y_test_proba = best_pipeline.predict_proba(X_test)[:, positive_class_index]

y_test_pred = (
    y_test_proba >= THRESHOLD
).astype(int)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1 Score": f1_score(y_test, y_test_pred),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba)
}

pd.DataFrame(
    test_metrics.items(),
    columns=["Metric", "Score"]
).round(4)

### 12.1 Classification Report

In [ ]:
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["No Subscription", "Subscription"]
))

### 12.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=["Predicted No", "Predicted Yes"],
        y=["Actual No", "Actual Yes"],
        text=cm,
        texttemplate="%{text}",
        showscale=False
    )
)

fig.update_layout(
    title="Confusion Matrix",
    xaxis_title="Prediction",
    yaxis_title="Actual"
)

fig.show()

### 12.3 ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"{best_model_name} | AUC = {roc_auc:.3f}"
    )
)

fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Random Baseline",
        line=dict(dash="dash")
    )
)

fig.update_layout(
    title="ROC Curve",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate"
)

fig.show()

## 13. Model Explainability: Feature Importance

In [ ]:
def get_feature_importance(fitted_pipeline):
    
    fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
    fitted_model = fitted_pipeline.named_steps["model"]
    
    feature_names = fitted_preprocessor.get_feature_names_out()
    
    if hasattr(fitted_model, "feature_importances_"):
        importance_values = fitted_model.feature_importances_
        
    elif hasattr(fitted_model, "coef_"):
        importance_values = np.abs(fitted_model.coef_[0])
        
    else:
        return None
    
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": importance_values
    })
    
    return importance_df.sort_values(
        "Importance",
        ascending=False
    )

feature_importance = get_feature_importance(best_pipeline)

display(feature_importance.head(15))

In [ ]:
top_features = (
    feature_importance
    .head(15)
    .sort_values("Importance")
)

fig = px.bar(
    top_features,
    x="Importance",
    y="Feature",
    orientation="h",
    title=f"Top 15 Important Features - {best_model_name}"
)

fig.show()

**Interpretation note:** These features were the most useful inputs for the selected model’s predictions. Feature importance indicates predictive association within this model; it does not prove that a feature causes subscription.

In [ ]:
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "bank_marketing_model.pkl"
metadata_path = MODELS_DIR / "model_metadata.pkl"

joblib.dump(best_pipeline, model_path)

metadata = {
    "model_name": best_model_name,
    "features": FEATURES,
    "threshold": THRESHOLD,
    "test_metrics": test_metrics
}

joblib.dump(metadata, metadata_path)

print("Model saved successfully ✅")
print(model_path.resolve())

## 14. Deployment Readiness

The saved file `bank_marketing_model.pkl` contains the full preprocessing-and-model pipeline. The Streamlit application (`app.py`) can therefore accept raw customer inputs and apply the same imputation, encoding, scaling, and prediction logic used during training.

## 15. Conclusion, Business Recommendation, and Limitations

### Model Result
Random Forest was selected as the best model based on cross-validation ROC-AUC (0.7952). On the held-out test set, it achieved:

- **Accuracy:** 0.8596
- **Precision:** 0.4192
- **Recall:** 0.6401
- **F1-score:** 0.5066
- **ROC-AUC:** 0.8147

### Business Recommendation
Use the model’s predicted probability to prioritize marketing outreach. A lower threshold increases Recall and identifies more potential subscribers, while a higher threshold concentrates effort on fewer, higher-confidence leads. The final threshold should be chosen using business capacity and the cost of marketing contacts.

### Limitations
1. The analysis is based on historical data and may not generalize perfectly to future campaigns.
2. Feature importance indicates predictive association, not causation.
3. The deployed model intentionally excludes `duration` to prevent leakage, so production performance can differ from models that include post-call information.
4. Threshold selection should be refined with the bank’s actual contact cost and conversion value.